# Load & Analyze Pass Event Graphs

Load the pre-processed graphs from `02_graph_construction` and explore their structure.

**Contents:**
1. Load saved graph dataset
2. Explore graph statistics
3. Analyze node and edge features
4. Visualize feature distributions
5. Prepare train/test split for GNN

In [ ]:
import os
import sys

# Setup workspace
workspace_root = os.path.expanduser("~/projects/football-analysis")
os.chdir(workspace_root)
sys.path.insert(0, workspace_root)

print(f"✓ Working directory: {os.getcwd()}")

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from torch_geometric.loader import DataLoader
from torch_geometric.data import Data

print(f"PyTorch version: {torch.__version__}")
print(f"Numpy version: {np.__version__}")

## 1. Load Saved Graphs

In [ ]:
# Load the dataset
dataset = torch.load("data/processed/metrica_game1_pass_graphs.pt")

print(f"✓ Loaded {len(dataset)} graphs")
print(f"\nDataset info:")
print(f"  Total graphs: {len(dataset)}")
for i, g in enumerate(dataset[:3]):
    print(f"  Graph {i}: {g.num_nodes} nodes, {g.num_edges} edges, label={g.y.item()}")

## 2. Explore Graph Statistics

In [ ]:
# Collect statistics
num_nodes_list = []
num_edges_list = []
labels_list = []

for g in dataset:
    num_nodes_list.append(g.num_nodes)
    num_edges_list.append(g.num_edges)
    labels_list.append(g.y.item())

num_nodes_arr = np.array(num_nodes_list)
num_edges_arr = np.array(num_edges_list)
labels_arr = np.array(labels_list)

print("Graph Structure Statistics")
print(f"  Nodes per graph:")
print(f"    Mean: {num_nodes_arr.mean():.2f}")
print(f"    Std:  {num_nodes_arr.std():.2f}")
print(f"    Min:  {num_nodes_arr.min()}")
print(f"    Max:  {num_nodes_arr.max()}")

print(f"\n  Edges per graph:")
print(f"    Mean: {num_edges_arr.mean():.2f}")
print(f"    Std:  {num_edges_arr.std():.2f}")
print(f"    Min:  {num_edges_arr.min()}")
print(f"    Max:  {num_edges_arr.max()}")

print(f"\n  Pass Outcomes:")
successful = (labels_arr == 1.0).sum()
unsuccessful = (labels_arr == 0.0).sum()
print(f"    Successful passes: {successful} ({successful/len(labels_arr)*100:.1f}%)")
print(f"    Unsuccessful passes: {unsuccessful} ({unsuccessful/len(labels_arr)*100:.1f}%)")

## 3. Analyze Node & Edge Features

In [ ]:
# Extract all node features
all_node_features = []
for g in dataset:
    all_node_features.append(g.x.numpy())

all_node_features = np.vstack(all_node_features)

feature_names = [
    "x (pitch)", "y (pitch)", "vx (vel_x)", "vy (vel_y)",
    "team", "dist_atk_goal", "dist_def_goal", "angle_atk", "pressure"
]

print(f"Node Features Shape: {all_node_features.shape}")
print(f"\nNode Feature Statistics:")
print(f"{'Feature':<20} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 52)

for i, name in enumerate(feature_names):
    feat = all_node_features[:, i]
    print(f"{name:<20} {feat.mean():>10.3f} {feat.std():>10.3f} {feat.min():>10.3f} {feat.max():>10.3f}")

In [ ]:
# Extract all edge features
all_edge_features = []
for g in dataset:
    all_edge_features.append(g.edge_attr.numpy())

all_edge_features = np.vstack(all_edge_features)

edge_feature_names = ["distance", "distance_normalized", "angle", "same_team"]

print(f"Edge Features Shape: {all_edge_features.shape}")
print(f"\nEdge Feature Statistics:")
print(f"{'Feature':<25} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 57)

for i, name in enumerate(edge_feature_names):
    feat = all_edge_features[:, i]
    print(f"{name:<25} {feat.mean():>10.3f} {feat.std():>10.3f} {feat.min():>10.3f} {feat.max():>10.3f}")

## 4. Visualize Feature Distributions

In [ ]:
# Plot node feature distributions
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, name in enumerate(feature_names):
    ax = axes[i]
    feat = all_node_features[:, i]
    ax.hist(feat, bins=50, alpha=0.7, color="steelblue", edgecolor="black")
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.suptitle("Node Feature Distributions", fontsize=14, fontweight="bold", y=1.00)
plt.show()

In [ ]:
# Plot edge feature distributions
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i, name in enumerate(edge_feature_names):
    ax = axes[i]
    feat = all_edge_features[:, i]
    ax.hist(feat, bins=50, alpha=0.7, color="coral", edgecolor="black")
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.suptitle("Edge Feature Distributions", fontsize=14, fontweight="bold", y=1.00)
plt.show()

In [ ]:
# Plot pass outcome distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Count plot
counts = pd.Series(labels_arr).value_counts()
ax1.bar(["Unsuccessful", "Successful"], [counts[0.0], counts[1.0]], color=["#ef5350", "#66bb6a"], alpha=0.8, edgecolor="black", linewidth=2)
ax1.set_ylabel("Number of Passes", fontsize=11)
ax1.set_title("Pass Outcome Distribution", fontsize=12, fontweight="bold")
ax1.grid(alpha=0.3, axis="y")

# Percentage plot
sizes = [counts[0.0], counts[1.0]]
labels_pie = [f"Unsuccessful\n({sizes[0]}, {sizes[0]/len(labels_arr)*100:.1f}%)",
              f"Successful\n({sizes[1]}, {sizes[1]/len(labels_arr)*100:.1f}%)"]
ax2.pie(sizes, labels=labels_pie, colors=["#ef5350", "#66bb6a"], autopct="", startangle=90, textprops={"fontsize": 10})
ax2.set_title("Pass Outcome Distribution (%)", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.show()

## 5. Prepare Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Split into train/val/test (70/15/15)
train_idx, test_idx = train_test_split(range(len(dataset)), test_size=0.3, random_state=42)
val_idx, test_idx = train_test_split(test_idx, test_size=0.5, random_state=42)

train_dataset = [dataset[i] for i in train_idx]
val_dataset = [dataset[i] for i in val_idx]
test_dataset = [dataset[i] for i in test_idx]

print(f"Dataset Split:")
print(f"  Train: {len(train_dataset)} graphs ({len(train_idx)/len(dataset)*100:.1f}%)")
print(f"  Val:   {len(val_dataset)} graphs ({len(val_idx)/len(dataset)*100:.1f}%)")
print(f"  Test:  {len(test_dataset)} graphs ({len(test_idx)/len(dataset)*100:.1f}%)")

# Check class balance in each split
for name, subset_idx in [("Train", train_idx), ("Val", val_idx), ("Test", test_idx)]:
    labels = labels_arr[subset_idx]
    pos = (labels == 1.0).sum()
    neg = (labels == 0.0).sum()
    print(f"\n{name} - Successful: {pos} ({pos/len(labels)*100:.1f}%), Unsuccessful: {neg} ({neg/len(labels)*100:.1f}%)")

In [ ]:
# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"✓ DataLoaders created:")
print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches: {len(val_loader)}")
print(f"  Test batches: {len(test_loader)}")

# Example batch
batch = next(iter(train_loader))
print(f"\nExample batch:")
print(f"  Batch size: {batch.num_graphs}")
print(f"  Total nodes: {batch.num_nodes}")
print(f"  Total edges: {batch.num_edges}")
print(f"  Node features: {batch.x.shape}")
print(f"  Edge features: {batch.edge_attr.shape}")
print(f"  Labels: {batch.y.shape}")

In [ ]:
# Save loaders for use in model training
os.makedirs("data/processed", exist_ok=True)
torch.save({
    "train_dataset": train_dataset,
    "val_dataset": val_dataset,
    "test_dataset": test_dataset,
    "train_loader": train_loader,
    "val_loader": val_loader,
    "test_loader": test_loader,
}, "data/processed/metrica_dataloaders.pt")

print("✓ Saved dataloaders to data/processed/metrica_dataloaders.pt")